# Fase 1 — Modelo predictivo: detección de intrusiones en red (UNSW-NB15)

**Curso:** Modelos y Simulación de Sistemas I — UdeA, 2026-2\
**Equipo:** Daniel Felipe Cetina, Bryan Medrano, Adrian Espinosa

## 1. Descripción del problema y del dataset

- **Problema:** clasificar cada flujo de tráfico de red como **normal (0)** o **ataque (1)**. Clasificación binaria.
- **Motivación:** La automatización en la detección de intrusiones de red (IDS) es esencial para interceptar ciberataques en tiempo real a escala industrial. El volumen masivo y la velocidad del tráfico moderno hacen inviable el análisis manual, por ende, un IDS basado en Machine Learning puede alertar sobre patrones maliciosos, reducir la revisión manual y ayudar a priorizar la respuesta del equipo de seguridad.
- **Dataset:** UNSW-NB15 (Australian Centre for Cyber Security), versión curada `UNSW_NB15_training-set.csv`. Cada fila es un flujo de red descrito por variables de tráfico (duración, bytes, paquetes, TTL, protocolo, servicio, estado…).
- **Variable objetivo:** `label`.
- **Columnas excluidas:** `id` (identificador) y `attack_cat` (tipo de ataque: se deriva directamente del objetivo, usarla sería fuga de información).
- **Métrica principal:** F1 de la clase ataque, porque importa tanto no dejar pasar ataques (recall) como no generar falsas alarmas (precision).
- **Hallazgos del EDA y decisiones tomadas:** Se analizó un total de 82,332 registros. El dataset no presenta valores nulos en las variables de entrada (`NaN = 0%`). La clase objetivo muestra un ligero desbalance hacia tráfico de ataque (55.06% ataques vs. 44.94% normal). Se identificaron variables altamente correlacionadas con el objetivo (`sttl`, `swin`, `ct_dst_sport_ltm`) y un uso muy frecuente del marcador `"-"` en `service` (57.27%), el cual se tratará como una categoría válida sin ser eliminado.

> Este notebook se ejecuta de principio a fin con **Reiniciar y ejecutar todo**. Todas las transformaciones se ajustan **solo con train** dentro de un `Pipeline`, así que no hay fuga de información.


## 2. Configuración

Toda la configuración y las **columnas de entrada** están definidas aquí, en un único lugar. Estas listas se reutilizarán en `predict.py` (fase 2) y en la validación de la API (fase 3).

In [ ]:
import os
import sys
import json
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score)

# --- Datos ---
DATA_PATH = Path(os.environ.get("DATA_PATH", "../data/raw/UNSW_NB15_training-set.csv"))
TARGET = "label"
COLUMNAS_EXCLUIR = ["id", "attack_cat"]  # id = identificador; attack_cat = fuga del objetivo

# --- Columnas de entrada del modelo (contrato para fase-2 y fase-3) ---
COLS_CATEGORICAS = ["proto", "service", "state"]
COLS_NUMERICAS = [
    "dur", "spkts", "dpkts", "sbytes", "dbytes", "rate", "sttl", "dttl", "sload", "dload",
    "sloss", "dloss", "sinpkt", "dinpkt", "sjit", "djit", "swin", "stcpb", "dtcpb", "dwin",
    "tcprtt", "synack", "ackdat", "smean", "dmean", "trans_depth", "response_body_len",
    "ct_srv_src", "ct_state_ttl", "ct_dst_ltm", "ct_src_dport_ltm", "ct_dst_sport_ltm",
    "ct_dst_src_ltm", "is_ftp_login", "ct_ftp_cmd", "ct_flw_http_mthd", "ct_src_ltm",
    "ct_srv_dst", "is_sm_ips_ports",
]
COLS_ENTRADA = COLS_NUMERICAS + COLS_CATEGORICAS

# --- Experimento ---
RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 3
VERSION_MODELO = "v1"
MODELS_DIR = Path("models")
MAX_MB_MODELO = 50  # por encima de esto no conviene subirlo a GitHub

np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
print("Python", sys.version.split()[0], "| sklearn", sklearn.__version__, "| pandas", pd.__version__)


## 3. Carga de datos

El CSV se coloca manualmente en `data/raw/` (ver `data/README.md`). Si el notebook corre en Google Colab y el archivo no está, se pide subirlo.

In [ ]:
if not DATA_PATH.exists() and "google.colab" in sys.modules:
    from google.colab import files
    print(f"No se encontró {DATA_PATH}. Sube UNSW_NB15_training-set.csv:")
    subido = files.upload()
    DATA_PATH = Path(next(iter(subido)))

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Coloca el CSV en {DATA_PATH.resolve()} (ver data/README.md)")

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Archivo: {DATA_PATH} | {df.shape[0]:,} filas x {df.shape[1]} columnas")

# El contrato de columnas debe coincidir con el CSV
faltantes = set(COLS_ENTRADA + [TARGET]) - set(df.columns)
no_usadas = set(df.columns) - set(COLS_ENTRADA + [TARGET] + COLUMNAS_EXCLUIR)
assert not faltantes, f"Columnas esperadas que no están en el CSV: {faltantes}"
print("Columnas del CSV no usadas ni excluidas:", no_usadas or "ninguna")
df.head()


## 4. Análisis exploratorio (EDA)

### 4.1 Dimensiones y tipos

In [ ]:
print("Dimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes.astype(str).value_counts())
print("\nFilas duplicadas:", df.duplicated(subset=COLS_ENTRADA + [TARGET]).sum())


### 4.2 Valores nulos

Se separan los NaN reales de los marcadores de texto como `"-"`. En UNSW-NB15, `service = "-"` significa *servicio no identificado*, así que se trata como una categoría legítima y no como un nulo.

In [ ]:
nulos = pd.DataFrame({
    "NaN_%": df[COLS_ENTRADA].isna().mean() * 100,
    "marcador_-_%": pd.Series({c: (df[c].astype(str).str.strip() == "-").mean() * 100
                              for c in COLS_CATEGORICAS}),
}).fillna(0)
nulos = nulos[(nulos > 0).any(axis=1)].round(3)
print(nulos if not nulos.empty else "Sin nulos ni marcadores en las columnas de entrada")

en_rango = nulos.index[nulos["NaN_%"].between(0.1, 2)] if not nulos.empty else []
print("\nPredictoras con 0,1 %-2 % de NaN (requisito del curso):", list(en_rango) or "NINGUNA")


### 4.3 Distribución del objetivo

In [ ]:
dist = df[TARGET].value_counts()
print(pd.DataFrame({"n": dist, "%": (dist / len(df) * 100).round(2)}))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
dist.plot.bar(ax=ax[0], title="Distribución de label (0 = normal, 1 = ataque)", rot=0)
if "attack_cat" in df.columns:  # solo para entender el problema; NO se usa como predictora
    df["attack_cat"].value_counts().plot.barh(ax=ax[1], title="Tipos de ataque (attack_cat)")
plt.tight_layout()
plt.show()


### 4.4 Estadísticas descriptivas

In [ ]:
df[COLS_NUMERICAS].describe().T

In [ ]:
for col in COLS_CATEGORICAS:
    print(f"{col}: {df[col].nunique()} categorías | top 5: {df[col].value_counts().head(5).to_dict()}")


### 4.5 Gráficas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, COLS_CATEGORICAS):
    top = df[col].value_counts().head(10).index
    (df[df[col].isin(top)].groupby(col)[TARGET].mean().sort_values()
       .plot.barh(ax=ax, title=f"Proporción de ataques por {col} (top 10)"))
plt.tight_layout()
plt.show()

# Variables numéricas muy sesgadas: se muestran en escala log1p
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["sbytes", "dur", "sttl"]):
    sns.histplot(data=df, x=np.log1p(df[col].clip(lower=0)), hue=TARGET, bins=50,
                 stat="density", common_norm=False, ax=ax)
    ax.set_xlabel(f"log1p({col})")
plt.tight_layout()
plt.show()


In [ ]:
corr = df[COLS_NUMERICAS + [TARGET]].corr(numeric_only=True)
top_corr = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False).head(15)
print("Top 15 |correlación| con el objetivo:")
print(top_corr.round(3))

plt.figure(figsize=(10, 8))
sns.heatmap(df[top_corr.index.tolist() + [TARGET]].corr(), cmap="coolwarm", center=0)
plt.title("Correlaciones entre las 15 variables más asociadas al objetivo")
plt.show()


## 5. Separación train/test

**Antes de cualquier transformación.** Estratificada por el objetivo y con `random_state` fijo. Desde aquí, el conjunto de test solo se usa para la evaluación final.

In [ ]:
X = df[COLS_ENTRADA]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print("Proporción de ataques -> train:", round(y_train.mean(), 4), "| test:", round(y_test.mean(), 4))
